[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_02/MFM_ch2_big_data/MFM_ch2_big_data.ipynb)

# MFM_ch2_big_data

Case study, Martin and Nagel (2022), Market efficiency in the age of big data: the specification of their Sec. 7 applied to the 49 US industry portfolios from July 1926. Predictors: simple and squared returns at lags 2 to 120 months (238 predictors), demeaned and standardised across portfolios each month, equal weight per month. Rolling 20-year OLS gives the in-sample return r_IS and the out-of-sample portfolio return r_OOS (10-year moving average with two-standard-error bands, as in their Fig. 8); a full-sample ridge regression from 1971 with leave-one-year-out cross-validation gives the coefficients by lag (Fig. 7).

*Modelling Financial Markets (MFM), Chapter 2: Market Efficiency. Author: Daniel Traian Pele.*

In [ ]:
import io
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
GROUP_COL = {'Developed': MainBlue, 'Emerging/frontier': IDAred, 'Crypto': Amber, 'FX': Forest}

SEED = 42
TABLE_DIR = '.'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date, days per year)
MARKETS = {
    'sp500':  ('GSPC.INDX',     'S&P 500 (US)',          'Developed', '2000-01-01', 252),
    'stoxx':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Developed', '2000-01-01', 252),
    'nikkei': ('N225.INDX',     'Nikkei 225 (Japan)',    'Developed', '2000-01-01', 252),
    'hsi':    ('HSI.INDX',      'Hang Seng (HK)',        'Developed', '2000-01-01', 252),
    'bet':    ('BET',           'BET (Romania)',         'Emerging/frontier', '2000-01-01', 252),
    'wig20':  ('WIG20.INDX',    'WIG20 (Poland)',        'Emerging/frontier', '2000-01-01', 252),
    'bux':    ('BUX.INDX',      'BUX (Hungary)',         'Emerging/frontier', '2000-01-01', 252),
    'px':     ('PX.INDX',       'PX (Czechia)',          'Emerging/frontier', '2000-01-01', 252),
    'xu100':  ('XU100.INDX',    'BIST 100 (Turkey)',     'Emerging/frontier', '2000-01-01', 252),
    'bvsp':   ('BVSP.INDX',     'Bovespa (Brazil)',      'Emerging/frontier', '2000-01-01', 252),
    'mxx':    ('MXX.INDX',      'IPC (Mexico)',          'Emerging/frontier', '2000-01-01', 252),
    'nsei':   ('NSEI.INDX',     'Nifty 50 (India)',      'Emerging/frontier', '2000-01-01', 252),
    'ssec':   ('SSEC.INDX',     'Shanghai (China)',      'Emerging/frontier', '2000-01-01', 252),
    'btc':    ('BTC-USD.CC',    'Bitcoin',               'Crypto', '2014-09-17', 365),
    'eth':    ('ETH-USD.CC',    'Ethereum',              'Crypto', '2016-01-01', 365),
    'eurron': ('REF:EUR',       'EUR/RON (reference rate)', 'FX', '2005-07-01', 252),
}

LABELS = {k: v[1] for k, v in MARKETS.items()}
GROUPS = {k: v[2] for k, v in MARKETS.items()}
PERIODS = {k: v[4] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one series (local copy of the course data, otherwise the GitHub repository)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (BNR annual XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0, _ = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def complete_months(x):
    """Drop the incomplete final month (last observation before the last business day of the month):
    the monthly analyses and the turn-of-the-month effect use complete months only."""
    last = x.index[-1]
    if last + pd.offsets.BMonthEnd(0) != last:
        x = x[x.index < last.to_period('M').start_time]
    return x


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


rets = {k: log_returns(k) for k in MARKETS}
ORDER = list(MARKETS)
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

## Efficiency tests

In [ ]:
def variance_ratio(r, q):
    """Overlapping VR(q) with the Lo-MacKinlay (1988) bias corrections.

    Returns (VR, homoskedastic z, heteroskedasticity-robust z*).
    """
    x = np.asarray(r, dtype=float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = (e ** 2).sum() / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu        # overlapping q-period sums
    m = q * (T - q + 1) * (1 - q / T)
    var_c = (agg ** 2).sum() / m
    vr = var_c / var_a
    z_homo = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = (e ** 2).sum() ** 2
    theta = 0.0
    for j in range(1, q):
        delta = T * (e[j:] ** 2 * e[:-j] ** 2).sum() / den
        theta += (2 * (q - j) / q) ** 2 * delta
    z_star = np.sqrt(T) * (vr - 1) / np.sqrt(theta)
    return vr, z_homo, z_star


def chow_denning(r, qs=(2, 5, 10, 20)):
    """Chow-Denning multiple test: max |z*(q)|; p-value from the studentized maximum modulus (infinite df)."""
    zs = [variance_ratio(r, q)[2] for q in qs]
    zmax = np.max(np.abs(zs))
    p = 1 - (2 * stats.norm.cdf(zmax) - 1) ** len(qs)
    return zmax, p, zs


def runs_test(r):
    """Number of sign runs (zeros dropped), with its mean and variance under independence."""
    s = np.sign(np.asarray(r, dtype=float))
    s = s[s != 0]
    n1, n2 = (s > 0).sum(), (s < 0).sum()
    n = n1 + n2
    runs = 1 + (s[1:] != s[:-1]).sum()
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return runs, mean, z, 2 * (1 - stats.norm.cdf(abs(z)))


def robust_ljung_box(r, m=10):
    """Ljung-Box Q(m) and Q~(m) with the robust variance tau_k = sum e_t^2 e_{t-k}^2 / (sum e_t^2)^2."""
    e = np.asarray(r, dtype=float) - np.mean(r)
    T = len(e)
    s2 = (e ** 2).sum()
    q_lb, q_rob = 0.0, 0.0
    for k in range(1, m + 1):
        rho = (e[k:] * e[:-k]).sum() / s2
        tau = (e[k:] ** 2 * e[:-k] ** 2).sum() / s2 ** 2
        q_lb += T * (T + 2) * rho ** 2 / (T - k)
        q_rob += rho ** 2 / tau
    return q_lb, 1 - stats.chi2.cdf(q_lb, m), q_rob, 1 - stats.chi2.cdf(q_rob, m)


def _rs(x):
    y = np.cumsum(x - x.mean())
    s = x.std(ddof=0)
    return (y.max() - y.min()) / s if s > 0 else np.nan


def expected_rs(n):
    """E[R/S] for i.i.d. noise (Anis & Lloyd, 1976)."""
    if n <= 340:
        f = np.exp(gammaln((n - 1) / 2) - gammaln(n / 2)) / np.sqrt(np.pi)
    else:
        f = 1 / np.sqrt(n * np.pi / 2)
    i = np.arange(1, n)
    return f * np.sum(np.sqrt((n - i) / i))


def rs_hurst(r, min_n=10, n_sizes=20):
    """H from the slope of log(R/S) on log(n) (Hurst, 1951) and the Anis-Lloyd corrected H (slope of R/S - E[R/S], plus 0.5)."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    sizes = np.unique(np.logspace(np.log10(min_n), np.log10(T // 4), n_sizes).astype(int))
    rs, ers = [], []
    for n in sizes:
        k = T // n
        vals = [_rs(x[i * n:(i + 1) * n]) for i in range(k)]
        rs.append(np.nanmean(vals))
        ers.append(expected_rs(n))
    ls, lrs, lers = np.log(sizes), np.log(rs), np.log(ers)
    h = np.polyfit(ls, lrs, 1)[0]
    h_al = 0.5 + np.polyfit(ls, lrs - lers, 1)[0]
    return h, h_al, pd.DataFrame({'n': sizes, 'rs': rs, 'expected_rs': ers})


def lo_modified_rs(r, q=None):
    """Lo's (1991) statistic V = Q_T / sqrt(T); under H0 (no long memory) V lies in [0.809, 1.862] with 95% probability."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    e = x - x.mean()
    if q is None:                                   # Andrews (1991) bandwidth for an AR(1)
        rho = np.corrcoef(e[1:], e[:-1])[0, 1]
        q = int(np.floor((3 * T / 2) ** (1 / 3) * abs(2 * rho / (1 - rho ** 2)) ** (2 / 3)))
    s2 = (e ** 2).mean()
    for j in range(1, q + 1):
        w = 1 - j / (q + 1)
        s2 += 2 * w * (e[j:] * e[:-j]).sum() / T
    y = np.cumsum(e)
    Q = (y.max() - y.min()) / np.sqrt(s2)
    return Q / np.sqrt(T), q


def dfa_hurst(r, min_box=10, n_sizes=18, max_frac=0.25):
    """DFA-1 exponent: slope of log F(n) on log n for the cumulative profile of demeaned returns."""
    x = np.asarray(r, dtype=float)
    y = np.cumsum(x - x.mean())
    T = len(y)
    sizes = np.unique(np.logspace(np.log10(min_box), np.log10(int(T * max_frac)), n_sizes).astype(int))
    F = []
    for n in sizes:
        k = T // n
        seg = y[:k * n].reshape(k, n)
        t = np.arange(n)
        A = np.vstack([t, np.ones(n)]).T
        coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
        resid = seg.T - A @ coef
        F.append(np.sqrt((resid ** 2).mean()))
    return np.polyfit(np.log(sizes), np.log(F), 1)[0], pd.DataFrame({'n': sizes, 'F': F})


def rolling_stat(r, func, window=500, step=21):
    """Apply func on rolling windows of `window` observations, every `step` observations."""
    x = r.values
    idx, out = [], []
    for end in range(window, len(x) + 1, step):
        idx.append(r.index[end - 1])
        out.append(func(x[end - window:end]))
    return pd.Series(out, index=idx)

## Helper functions

In [ ]:
import zipfile

FF49_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/49_Industry_Portfolios_CSV.zip'

MN_LAGS, MN_WINDOW, MN_MA = 120, 240, 120

MN_RIDGE_START = '1971-01-01'

## Analysis

In [ ]:
def french_industries49():
    """49 Industry Portfolios (Kenneth French Data Library): value-weighted average monthly returns,
    as fractions; missing values (-99.99, -999) become NaN."""
    raw = urllib.request.urlopen(urllib.request.Request(FF49_URL, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    lines = z.read(z.namelist()[0]).decode('latin-1').splitlines()
    i0 = next(i for i, l in enumerate(lines) if 'Average Value Weighted Returns -- Monthly' in l) + 1
    i1 = next(i for i in range(i0 + 1, len(lines)) if not lines[i].strip())
    d = pd.read_csv(io.StringIO('\n'.join(lines[i0:i1])), index_col=0)
    d.columns = [c.strip() for c in d.columns]
    d.index = pd.to_datetime(d.index.astype(str).str.strip(), format='%Y%m') + pd.offsets.MonthEnd(0)
    d = d.apply(pd.to_numeric, errors='coerce')
    return d.where(d > -99) / 100


def mn_panel(R, lags=MN_LAGS):
    """For each month t: X_t (N_t x 238), the returns t-2..t-120 and their squares, demeaned and standardised
    across portfolios; y_t = r_t demeaned across portfolios. Returns the months, the X, the y and the portfolios used."""
    A = R.values
    months, XS, YS, NS = [], [], [], []
    for t in range(lags, len(A)):
        ok = ~np.isnan(A[t - lags:t + 1]).any(axis=0)            # complete history t-120 ... t
        if ok.sum() < 10:
            continue
        past = np.column_stack([A[t - k, ok] for k in range(2, lags + 1)])
        X = np.hstack([past, past ** 2])
        X = (X - X.mean(axis=0)) / X.std(axis=0)
        y = A[t, ok] - A[t, ok].mean()
        months.append(R.index[t])
        XS.append(X)
        YS.append(y)
        NS.append(int(ok.sum()))
    return pd.DatetimeIndex(months), XS, YS, np.array(NS)


def mn_moments(XS, YS):
    """Monthly weighted moments (weight 1/N_t): G_t = X'X/N, b_t = X'y/N, v_t = y'y/N."""
    G = np.array([X.T @ X / len(y) for X, y in zip(XS, YS)])
    b = np.array([X.T @ y / len(y) for X, y in zip(XS, YS)])
    v = np.array([y @ y / len(y) for y in YS])
    return G, b, v


def mn_rolling(months, XS, YS, G, b, v, window=MN_WINDOW, ma=MN_MA):
    """OLS on rolling 20-year windows: r_IS (in sample) and r_OOS,t+1 (Martin & Nagel, eq. 19; Fig. 8)."""
    rows = []
    for e in range(window - 1, len(months) - 1):
        Gs, bs = G[e - window + 1:e + 1].sum(axis=0), b[e - window + 1:e + 1].sum(axis=0)
        h = np.linalg.solve(Gs, bs)
        r_is = h @ bs / window                                   # mean of (1/N) h'X_s'y_s in the window
        X1, y1 = XS[e + 1], YS[e + 1]
        r2_is = r_is / v[e - window + 1:e + 1].mean()           # in-sample R^2 of the window
        rows.append((months[e + 1], r_is, r2_is, (X1 @ h) @ y1 / len(y1)))
    o = pd.DataFrame(rows, columns=['date', 'r_is', 'r2_is', 'r_oos']).set_index('date')
    o['oos_ma'] = o['r_oos'].rolling(ma).mean()
    o['oos_se'] = np.sqrt((o['r_oos'] ** 2).rolling(ma).mean() / ma)
    return o


def mn_ridge(months, G, b, v, start=MN_RIDGE_START, grid=np.logspace(0, 7, 57)):
    """Full-sample ridge from `start`; penalty: maximises the mean R^2 of the left-out years (leave-one-year-out)."""
    sel = months >= start
    yrs = months.year[sel]
    Gs, bs, vs = G[sel], b[sel], v[sel]
    U = np.unique(yrs)
    Gy = {u: Gs[yrs == u].sum(axis=0) for u in U}
    by = {u: bs[yrs == u].sum(axis=0) for u in U}
    vy = {u: vs[yrs == u].sum() for u in U}
    GT, bT, I = sum(Gy.values()), sum(by.values()), np.eye(G.shape[1])

    def cv_r2(lam):
        r2 = []
        for u in U:
            beta = np.linalg.solve(GT - Gy[u] + lam * I, bT - by[u])
            sse = vy[u] - 2 * beta @ by[u] + beta @ Gy[u] @ beta
            r2.append(1 - sse / vy[u])
        return float(np.mean(r2))

    cv = np.array([cv_r2(l) for l in grid])
    lam = float(grid[cv.argmax()])
    beta = np.linalg.solve(GT + lam * I, bT)
    L = G.shape[1] // 2
    coef = pd.DataFrame({'lag': np.arange(2, L + 2), 'b_ret': beta[:L], 'b_sq': beta[L:]})
    return coef, dict(lam=lam, cv_r2=float(cv.max()), cv_r2_ols=cv_r2(1e-8), start=str(months[sel][0].date()),
                      end=str(months[sel][-1].date()), years=len(U))


def fig_mn_rolling(o):
    """Martin & Nagel Fig. 8 on 49 portfolios: 10-year moving average of r_OOS with 2-standard-error bands, and r_IS."""
    x = o.dropna()
    s = 1e5
    fig, ax = plt.subplots(figsize=(7.2, 3.3))
    ax.fill_between(x.index, s * (x['oos_ma'] - 2 * x['oos_se']), s * (x['oos_ma'] + 2 * x['oos_se']),
                    color=MainBlue, alpha=0.18, lw=0, label=r'$\pm 2$ standard errors')
    ax.plot(x.index, s * x['oos_ma'], color=MainBlue, lw=1.5,
            label=r'$r_{OOS}$: out-of-sample, 10-year moving average')
    ax.plot(x.index, s * x['r_is'], color=IDAred, lw=1.3, label=r'$r_{IS}$: in-sample, 20-year window')
    ax.axhline(0, color=Gray, lw=0.7)
    ax.set_ylabel(r'Return per month ($\times 10^{-5}$)')
    ax.set_title('49 US industry portfolios, 238 lagged-return predictors: in-sample vs out-of-sample',
                 fontsize=9, loc='left')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=7.5)
    plt.tight_layout()
    save_fig('ch2_mn_rolling')


def fig_mn_ridge(coef):
    """Martin & Nagel Fig. 7 on 49 portfolios: ridge coefficients on past returns and on their squares."""
    lag = coef['lag'].values
    m12 = lag % 12 == 0
    short = (lag <= 12) & ~m12
    col = np.where(m12, IDAred, np.where(short, Forest, MainBlue))
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), sharex=True)
    axes[0].bar(lag, 1e5 * coef['b_ret'], color=col, width=0.8)
    axes[0].set_title('(a) Past returns', fontsize=9, loc='left')
    axes[1].bar(lag, 1e5 * coef['b_sq'], color=Amber, width=0.8)
    axes[1].set_title('(b) Past squared returns', fontsize=9, loc='left')
    for ax in axes:
        ax.axhline(0, color=Gray, lw=0.6)
        ax.set_xlabel('Lag (months)')
    axes[0].set_ylabel(r'Ridge coefficient ($\times 10^{-5}$)')
    handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=l) for c, l in
               [(Forest, 'Lags 2-11'), (IDAred, 'Multiples of 12'), (MainBlue, 'Other lags above 12'),
                (Amber, 'Squared returns')]]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False)
    plt.tight_layout()
    save_fig('ch2_mn_ridge')


def mn_summary(o, coef, ridge, months, NS, R):
    """The numbers shown on the slides."""
    x = o.dropna()
    t = x['oos_ma'] / x['oos_se']
    above = t[t > 2]
    se_all = np.sqrt((o['r_oos'] ** 2).mean() / len(o))
    half = o.index[0] + pd.DateOffset(years=30)
    early, late = o.loc[:half - pd.DateOffset(days=1)], o.loc[half:]
    lag = coef['lag']
    m12, short = lag % 12 == 0, (lag <= 11)
    other = (lag > 12) & ~m12
    return dict(
        data_start=str(R.index[0].date()), data_end=str(R.index[-1].date()),
        panel_start=str(months[0].date()), n_min=int(NS.min()), n_max=int(NS.max()), n_last=int(NS[-1]),
        oos_start=str(o.index[0].date()), oos_end=str(o.index[-1].date()), n_oos=len(o),
        r_is_mean=float(o['r_is'].mean()), r_is_min=float(o['r_is'].min()), r_is_max=float(o['r_is'].max()),
        r2_is=float(o['r2_is'].mean()),
        r_oos_mean=float(o['r_oos'].mean()), r_oos_t=float(o['r_oos'].mean() / se_all),
        early_end=str(early.index[-1].date()), early_mean=float(early['r_oos'].mean()),
        early_t=float(early['r_oos'].mean() / np.sqrt((early['r_oos'] ** 2).mean() / len(early))),
        late_mean=float(late['r_oos'].mean()),
        late_t=float(late['r_oos'].mean() / np.sqrt((late['r_oos'] ** 2).mean() / len(late))),
        share_above_2se=float((t > 2).mean()), last_above_2se=str(above.index[-1].date()) if len(above) else '',
        ma_last=float(x['oos_ma'].iloc[-1]), ma_last_t=float(t.iloc[-1]), ma_first=str(x.index[0].date()),
        max_t_2014_2019=float(t.loc['2014':'2019'].max()),
        ridge=ridge,
        b_short_mean=float(coef.loc[short, 'b_ret'].mean()), b_short_pos=int((coef.loc[short, 'b_ret'] > 0).sum()),
        b_m12_mean=float(coef.loc[m12, 'b_ret'].mean()), b_m12_pos=int((coef.loc[m12, 'b_ret'] > 0).sum()),
        n_m12=int(m12.sum()),
        b_other_mean=float(coef.loc[other, 'b_ret'].mean()), b_other_neg=float((coef.loc[other, 'b_ret'] < 0).mean()),
        sq_above50_pos=float((coef.loc[lag > 50, 'b_sq'] > 0).mean()))

## Run

In [ ]:
R = french_industries49()
months, XS, YS, NS = mn_panel(R)
G, b, v = mn_moments(XS, YS)
o = mn_rolling(months, XS, YS, G, b, v)
coef, ridge = mn_ridge(months, G, b, v)
fig_mn_rolling(o)
fig_mn_ridge(coef)
mn_summary(o, coef, ridge, months, NS, R)

![ch2_mn_rolling](./ch2_mn_rolling.png)

Output: `ch2_mn_rolling.pdf`

![ch2_mn_ridge](./ch2_mn_ridge.png)

Output: `ch2_mn_ridge.pdf`